In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from scipy.optimize import curve_fit
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader

from analysis.gravnet.model import NeutrinoGravNetRegressionFASER
from analysis.utils.utils import get_torch_path, get_weights_path, get_figures_path

WEIGHTS_DIR = "gravnet_regression_faser_all_events_mean_log"
LOG_TARGETS = True
RUN = 10000
DATA_TYPE = "all"
BATCH_SIZE = 8

TARGET_NAMES = ["E_nu", "E_lepton", "E_roe"]
TARGET_LATEX = [r"$E_\nu$", r"$E_\mathrm{lep}$", r"$E_\mathrm{roe}$"]
TARGET_UNITS = "TeV"

ENERGY_BINS_TEV = [
    (0.01, 0.05), (0.05, 0.1), (0.1, 0.2),
    (0.2,  0.3),  (0.3,  0.5), (0.5,  0.7),
    (0.7,  1.0),  (1.0,  1.5), (1.5,  3.0),
]

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

weights_path = get_weights_path() / WEIGHTS_DIR
torch_path   = get_torch_path()
figures_path = get_figures_path() / WEIGHTS_DIR
figures_path.mkdir(parents=True, exist_ok=True)

print(f"Weights : {weights_path}")
print(f"Figures : {figures_path}")

**Training curves**

In [ ]:
metrics = np.load(weights_path / "training_metrics.npz")
epochs  = np.arange(1, len(metrics["train_loss"]) + 1)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

ax = axes[0, 0]
ax.plot(epochs, metrics["train_loss"], label="Train")
ax.plot(epochs, metrics["val_loss"],   label="Val")
ax.set_yscale("log")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE loss")
ax.legend()
ax.set_title("MSE loss  (log10 TeV)")

ax = axes[0, 1]
ax.plot(epochs, metrics["train_rmse"], label="Train")
ax.plot(epochs, metrics["val_rmse"],   label="Val")
ax.set_xlabel("Epoch")
ax.set_ylabel("RMSE")
ax.legend()
ax.set_title("RMSE")

ax = axes[1, 0]
for i, latex in enumerate(TARGET_LATEX):
    ax.plot(epochs, metrics["val_rel_err"][:, i], label=latex)
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean |pred - true| / true")
ax.legend()
ax.set_title("Val relative error")

ax = axes[1, 1]
for i, latex in enumerate(TARGET_LATEX):
    ax.plot(epochs, metrics["val_resolution"][:, i], label=latex)
ax.set_xlabel("Epoch")
ax.set_ylabel("std(rel. error)")
ax.legend()
ax.set_title("Val resolution (std)")

plt.tight_layout()
plt.savefig(figures_path / "training_curves.png", dpi=300)
plt.show()
print("Saved: training_curves.png")

**Model and data**

In [ ]:
# Load checkpoint
checkpoint = torch.load(weights_path / "best_model.pt", weights_only=False)
print(f"Best model from epoch {checkpoint['epoch'] + 1},  "
      f"val_loss={checkpoint['val_loss']:.4f},  "
      f"val_rmse={checkpoint['val_rmse']:.4f}")

# Reconstruct model — args must match training exactly
model = NeutrinoGravNetRegressionFASER(
    input_dim=1,      # log10(n_hits) per super-pixel
    num_targets=3,    # E_nu, E_lepton, E_roe
    faser_dim=5,      # nhits_0, nhits_1, nhits_2, faser_x, faser_y
    pooling="mean",
)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()
print("Model loaded and set to eval mode.")

# Load the same data as training and reproduce the identical val split.
# train_test_split(random_state=42) is deterministic so we get the same 20%.
run_str  = "nue"   # run 10000 % 4 == 0 → nue
run_path = torch_path / f"{RUN}/pointnetpp_faser_{DATA_TYPE}_events"
chunk_files = sorted(run_path.glob(f"{run_str}_*.pt"))

dataset = []
for f in chunk_files:
    dataset.extend(torch.load(f, weights_only=False))
print(f"Loaded {len(dataset)} events from {len(chunk_files)} chunks.")

_, val_dataset = train_test_split(dataset, test_size=0.2, random_state=42)
print(f"Val set: {len(val_dataset)} events.")

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

**Inference**

Model outputs are in log10(TeV). Targets in the dataset are linear TeV.

In [ ]:
all_preds   = []
all_targets = []

with torch.no_grad():
    for data in val_loader:
        data    = data.to(device)
        preds   = model(data.x, data.pos, data.batch, data.x_faser)
        targets = torch.stack([data.E_nu, data.E_lepton, data.E_roe], dim=1)
        all_preds.append(preds.cpu())
        all_targets.append(targets.cpu())

preds_arr   = torch.cat(all_preds).numpy()    # [N, 3]  log10(TeV) if LOG_TARGETS
targets_arr = torch.cat(all_targets).numpy()  # [N, 3]  TeV

if LOG_TARGETS:
    preds_linear   = 10 ** preds_arr
    targets_linear = targets_arr
else:
    preds_linear   = preds_arr
    targets_linear = targets_arr

print(f"Inference done: {len(preds_linear)} events")
for i, name in enumerate(TARGET_NAMES):
    print(f"  {name:12s}: true [{targets_linear[:, i].min():.4f}, "
          f"{targets_linear[:, i].max():.4f}] TeV")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (name, latex) in enumerate(zip(TARGET_NAMES, TARGET_LATEX)):
    axes[i].hist(targets_linear[:, i], bins=50, histtype="step", linewidth=1.5)
    axes[i].set_xlabel(f"True {latex} [TeV]")
    axes[i].set_ylabel("Events")
    axes[i].set_title(name)
plt.suptitle("True energy distributions", fontsize=11)
plt.tight_layout()
plt.show()

**True vs reconstructed energy**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, latex) in enumerate(zip(TARGET_NAMES, TARGET_LATEX)):
    ax     = axes[i]
    y_true = targets_linear[:, i]
    y_pred = preds_linear[:, i]

    # Log-spaced bins spanning the full range of both true and predicted
    lo   = max(min(y_true.min(), y_pred.min()), 1e-6)
    hi   = max(y_true.max(), y_pred.max())
    bins = np.logspace(np.log10(lo), np.log10(hi), 100)

    h, _, _, image = ax.hist2d(
        y_true, y_pred,
        bins=[bins, bins],
        norm="log",
        cmap="viridis",
    )
    plt.colorbar(image, label="Events", ax=ax)

    # Perfect-reconstruction reference line
    ax.plot([lo, hi], [lo, hi], "r--", linewidth=1, label="y = x")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(rf"True {latex} [{TARGET_UNITS}]")
    ax.set_ylabel(rf"Reconstructed {latex} [{TARGET_UNITS}]")
    ax.legend(fontsize=8)
    ax.set_title(name)

plt.tight_layout()
plt.savefig(figures_path / "true_vs_reco.png", dpi=300)
plt.show()
print("Saved: true_vs_reco.png")

**Resolution histograms per energy bin**

Resolution = (pred − true) / true per event. A value of +0.3 means the prediction was 30% too high. These histograms show the full distribution shape in each energy bin — not just the width, but also symmetry and tail behaviour. The Gaussian fit gives mu (systematic offset) and sigma (spread).

In [ ]:
def gauss(x, A, mu, sigma):
    return A * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

for target_idx, (name, latex) in enumerate(zip(TARGET_NAMES, TARGET_LATEX)):
    y_true     = targets_linear[:, target_idx]
    y_pred     = preds_linear[:, target_idx]
    resolution = (y_pred - y_true) / y_true

    fig, axes_grid = plt.subplots(3, 3, figsize=(12, 10), constrained_layout=True)
    axes_flat = axes_grid.flatten()

    for j, (emin, emax) in enumerate(ENERGY_BINS_TEV):
        if j >= 9:
            break
        ax      = axes_flat[j]
        mask    = (y_true >= emin) & (y_true < emax)
        res_bin = resolution[mask]

        if len(res_bin) == 0:
            ax.text(0.5, 0.5, "No events", ha="center", va="center",
                    transform=ax.transAxes)
            ax.set_title(f"{emin}-{emax} TeV")
            continue

        p1, p99  = np.percentile(res_bin, [1, 99])
        xlim     = min(max(abs(p1), abs(p99), 0.1), 5.0)
        bins_arr = np.linspace(-xlim, xlim, 60)

        counts, bin_edges, _ = ax.hist(
            res_bin, bins=bins_arr, histtype="step", linewidth=1.5
        )

        bin_centers_fit = 0.5 * (bin_edges[:-1] + bin_edges[1:])
        try:
            popt_g, _ = curve_fit(
                gauss, bin_centers_fit, counts,
                p0=[counts.max(), np.mean(res_bin), np.std(res_bin)],
                maxfev=5000,
            )
            sigma_fit = abs(popt_g[2])
            x_gauss   = np.linspace(-xlim, xlim, 300)
            ax.plot(x_gauss, gauss(x_gauss, *popt_g), "r-", linewidth=1.5,
                    label="Gauss fit")
            label_text = f"mu = {popt_g[1]:.3f}\nsigma = {sigma_fit:.3f}"
            ax.legend(fontsize=7)
        except (RuntimeError, ValueError):
            label_text = f"mean = {np.mean(res_bin):.3f}\nstd  = {np.std(res_bin):.3f}"

        ax.set_title(f"{emin}-{emax} TeV  (N={len(res_bin)})")
        ax.set_xlabel("(pred - true) / true")
        ax.set_ylabel("Events")
        ax.set_xlim(-xlim, xlim)
        ax.text(
            0.05, 0.95, label_text,
            transform=ax.transAxes, va="top", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7),
        )

    fig.suptitle(f"{name} resolution per energy bin", fontsize=14)
    plt.savefig(figures_path / f"{name}_ResolutionPerBin.png", dpi=300)
    plt.show()
    print(f"Saved: {name}_ResolutionPerBin.png")

**Resolution vs energy**

std of (pred − true) / true in each energy bin. Should decrease with energy as higher-energy showers produce more hits and are easier to reconstruct.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for target_idx, (name, latex) in enumerate(zip(TARGET_NAMES, TARGET_LATEX)):
    ax         = axes[target_idx]
    y_true     = targets_linear[:, target_idx]
    y_pred     = preds_linear[:, target_idx]
    resolution = (y_pred - y_true) / y_true

    bin_centers = []
    sigmas      = []
    sigma_errs  = []

    for emin, emax in ENERGY_BINS_TEV:
        mask    = (y_true >= emin) & (y_true < emax)
        res_bin = resolution[mask]

        if len(res_bin) < 50:   # skip underpopulated bins
            continue

        sigma     = np.std(res_bin)
        sigma_err = sigma / np.sqrt(2 * len(res_bin))   # error on sigma
        center    = np.mean(y_true[mask])

        bin_centers.append(center)
        sigmas.append(sigma)
        sigma_errs.append(sigma_err)

    if bin_centers:
        ax.errorbar(bin_centers, sigmas, yerr=sigma_errs, fmt="o", capsize=3)

    # Mark bins where sigma >= 1 (reconstruction worse than random)
    for x, s in zip(bin_centers, sigmas):
        if s >= 1.0:
            ax.annotate("", xy=(x, 1.0), xytext=(x, 0.92),
                        arrowprops=dict(arrowstyle="->", linewidth=1.5, color="red"))

    ax.set_xscale("log")
    ax.set_ylim(0, 1.05)
    ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
    ax.set_xlabel(rf"True {latex} [{TARGET_UNITS}]")
    ax.set_ylabel(rf"{latex} Resolution (\u03c3)")
    ax.set_title(name)

plt.tight_layout()
plt.savefig(figures_path / "resolution_vs_true.png", dpi=300)
plt.show()
print("Saved: resolution_vs_true.png")

**Bias vs energy**

Mean of (pred − true) / true per energy bin. Should be near zero. A non-zero slope means the model systematically over- or under-estimates at certain energies.

In [ ]:
def line_log(x, m, c):
    """Bias model: linear in log10(E)."""
    return m * np.log10(x) + c

fig, axes   = plt.subplots(1, 3, figsize=(18, 5))
bias_params = {}

for target_idx, (name, latex) in enumerate(zip(TARGET_NAMES, TARGET_LATEX)):
    ax         = axes[target_idx]
    y_true     = targets_linear[:, target_idx]
    y_pred     = preds_linear[:, target_idx]
    resolution = (y_pred - y_true) / y_true

    bin_centers = []
    means       = []
    mean_errs   = []
    ns          = []

    for emin, emax in ENERGY_BINS_TEV:
        mask    = (y_true >= emin) & (y_true < emax)
        res_bin = resolution[mask]
        n       = len(res_bin)
        if n < 50:
            continue
        sigma = np.std(res_bin)
        bin_centers.append(np.mean(y_true[mask]))
        means.append(np.mean(res_bin))
        mean_errs.append(sigma / np.sqrt(n))
        ns.append(n)

    if bin_centers:
        ax.errorbar(bin_centers, means, yerr=mean_errs, fmt="o", capsize=3)

    if len(bin_centers) >= 2:
        try:
            popt, _ = curve_fit(
                line_log, bin_centers, means,
                sigma=mean_errs, absolute_sigma=True,
            )
            x_fit = np.logspace(
                np.log10(min(bin_centers)), np.log10(max(bin_centers)), 100
            )
            ax.plot(x_fit, line_log(x_fit, *popt), "r--",
                    label=f"fit: {popt[0]:.2e} * log10(E) + {popt[1]:.2e}")
            ax.legend(fontsize=8)
            bias_params[name] = popt
        except RuntimeError:
            pass

    ax.axhline(0, color="k", linestyle=":", linewidth=0.8, alpha=0.5)
    ax.set_xscale("log")
    ax.set_xlabel(f"True {latex} [{TARGET_UNITS}]")
    ax.set_ylabel("Bias  (mean +/- std/sqrt(N))")
    ax.set_title(name)

plt.tight_layout()
plt.savefig(figures_path / "bias.png", dpi=300)
plt.show()
print("Saved: bias.png")

**Summary**

In [ ]:
print(f"Best model : epoch {checkpoint['epoch'] + 1}")
print(f"Val MSE    : {checkpoint['val_loss']:.4f}")
print(f"Val RMSE   : {checkpoint['val_rmse']:.4f}")
print()
print(f"{'Target':<12} {'Mean RelErr':>12} {'Std(res)':>10} {'Bias (mean)':>12}")

for i, name in enumerate(TARGET_NAMES):
    y_true  = targets_linear[:, i]
    y_pred  = preds_linear[:, i]
    rel_err = np.abs(y_pred - y_true) / y_true
    res     = (y_pred - y_true) / y_true
    print(f"{name:<12} {rel_err.mean():>12.3f} {res.std():>10.3f} {res.mean():>+12.3f}")

print()
print(f"Figures: {figures_path}")

**Baseline: total calorimeter hits**

Each graph node stores log10(n_hits) for one calorimeter super-pixel. Summing 10^x over all nodes gives total raw hits — the same quantity as `len(hit_colID)` in `explore_tau_dataset.ipynb`, which correlates strongly with E_nu. A linear fit `E = a * total_hits + b` is the simplest possible reconstruction. GravNet should beat it, especially for E_lep and E_roe where the spatial shower structure matters.

In [ ]:
# ── Total calorimeter hits per event ─────────────────────────────────────────
# data.x = [N_nodes, 1], values are log10(n_hits) per super-pixel
# → 10^x gives hits per super-pixel; summing recovers total calorimeter hits
hits_list = []
for data in val_dataset:
    hits_list.append((10 ** data.x).sum().item())

total_hits_arr = np.array(hits_list)   # [N_events]

# ── Linear fit for each target: E_target = a * total_hits + b ────────────────
coeffs_all       = []
linear_preds_all = []
for i in range(3):
    c = np.polyfit(total_hits_arr, targets_linear[:, i], 1)
    coeffs_all.append(c)
    linear_preds_all.append(np.polyval(c, total_hits_arr))

linear_res_all  = [
    (linear_preds_all[i] - targets_linear[:, i]) / targets_linear[:, i]
    for i in range(3)
]
gravnet_res_all = [
    (preds_linear[:, i] - targets_linear[:, i]) / targets_linear[:, i]
    for i in range(3)
]

# ── Figure 1: Hits scatter + fit (one panel per target) ──────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, (name, latex) in enumerate(zip(TARGET_NAMES, TARGET_LATEX)):
    ax = axes[i]
    ax.scatter(total_hits_arr, targets_linear[:, i], s=2, alpha=0.3)
    x_fit = np.linspace(total_hits_arr.min(), total_hits_arr.max(), 100)
    ax.plot(x_fit, np.polyval(coeffs_all[i], x_fit), "r-", linewidth=2,
            label=f"r = {np.corrcoef(total_hits_arr, targets_linear[:, i])[0,1]:.3f}")
    ax.set_xlabel("Total calorimeter hits")
    ax.set_ylabel(rf"True {latex} [TeV]")
    ax.set_title(name)
    ax.legend(fontsize=9)
plt.suptitle("Baseline: total calorimeter hits vs energy targets", fontsize=12)
plt.tight_layout()
plt.savefig(figures_path / "baseline_scatter.png", dpi=300)
plt.show()
print("Saved: baseline_scatter.png")

# ── Figure 2: Residual distributions, all 3 targets ──────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, (name, latex) in enumerate(zip(TARGET_NAMES, TARGET_LATEX)):
    ax      = axes[i]
    gn_res  = gravnet_res_all[i]
    lin_res = linear_res_all[i]
    xlim    = min(np.percentile(np.abs(np.concatenate([gn_res, lin_res])), 99), 5.0)
    bins    = np.linspace(-xlim, xlim, 60)
    ax.hist(gn_res,  bins=bins, histtype="step", linewidth=1.5,
            label=f"GravNet  std={np.std(gn_res):.2f}")
    ax.hist(lin_res, bins=bins, histtype="step", linewidth=1.5, linestyle="--",
            label=f"Linear   std={np.std(lin_res):.2f}")
    ax.set_xlabel(
        rf"$({latex}^{{\rm reco}} - {latex}^{{\rm true}}) / {latex}^{{\rm true}}$"
    )
    ax.set_ylabel("Events")
    ax.set_title(name)
    ax.legend(fontsize=8)
plt.suptitle("Residuals: GravNet vs linear baseline", fontsize=12)
plt.tight_layout()
plt.savefig(figures_path / "baseline_residuals.png", dpi=300)
plt.show()
print("Saved: baseline_residuals.png")

# ── Figure 3: Std(res) vs E, all 3 targets ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, (name, latex) in enumerate(zip(TARGET_NAMES, TARGET_LATEX)):
    ax      = axes[i]
    gn_res  = gravnet_res_all[i]
    lin_res = linear_res_all[i]
    gn_sigmas, lin_sigmas, centers_b = [], [], []
    for emin, emax in ENERGY_BINS_TEV:
        mask = (targets_linear[:, i] >= emin) & (targets_linear[:, i] < emax)
        if mask.sum() < 20:
            continue
        centers_b.append(np.mean(targets_linear[mask, i]))
        gn_sigmas.append(np.std(gn_res[mask]))
        lin_sigmas.append(np.std(lin_res[mask]))
    ax.plot(centers_b, gn_sigmas,  "o-",  label="GravNet")
    ax.plot(centers_b, lin_sigmas, "s--", label="Linear fit")
    ax.set_xscale("log")
    ax.set_xlabel(rf"True {latex} [TeV]")
    ax.set_ylabel(r"Resolution ($\sigma$)")
    ax.set_title(name)
    ax.legend()
plt.suptitle("Resolution vs energy: GravNet vs linear baseline", fontsize=12)
plt.tight_layout()
plt.savefig(figures_path / "baseline_resolution.png", dpi=300)
plt.show()
print("Saved: baseline_resolution.png")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'Target':<12} {'GravNet std':>12} {'Linear std':>11} {'GravNet mean':>14} {'Linear mean':>12}")
for i, name in enumerate(TARGET_NAMES):
    print(f"{name:<12} {np.std(gravnet_res_all[i]):>12.3f} {np.std(linear_res_all[i]):>11.3f} "
          f"{np.mean(gravnet_res_all[i]):>+14.3f} {np.mean(linear_res_all[i]):>+12.3f}")

**Inelasticity and energy conservation**

Inelasticity y = E_lep / E_nu (fraction of neutrino energy carried by the lepton, 0–1 for CC events).

The true labels satisfy energy conservation: (E_lep + E_roe) / E_nu ≈ 1. The conservation plot checks whether the model respects this — predictions that are internally inconsistent will have a mean far from 1.

In [ ]:
E_nu_true  = targets_linear[:, 0]
E_lep_true = targets_linear[:, 1]
E_roe_true = targets_linear[:, 2]
E_nu_pred  = preds_linear[:, 0]
E_lep_pred = preds_linear[:, 1]
E_roe_pred = preds_linear[:, 2]

inel_true = E_lep_true / E_nu_true.clip(min=1e-6)
inel_pred = E_lep_pred / E_nu_pred.clip(min=1e-6)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# inelasticity true vs predicted
ax     = axes[0]
hi_i   = min(max(np.percentile(inel_true, 99), np.percentile(inel_pred, 99)), 3.0)
bins_i = np.linspace(0.0, hi_i, 80)
h, _, _, img = ax.hist2d(inel_true, inel_pred, bins=[bins_i, bins_i],
                          norm="log", cmap="viridis")
plt.colorbar(img, ax=ax, label="Events")
ax.plot([0, hi_i], [0, hi_i], "r--", linewidth=1, label="y = x")
ax.set_xlabel("True $y$")
ax.set_ylabel("Predicted $y$")
ax.set_title("Inelasticity $y = E_{lep} / E_\\nu$")
ax.legend(fontsize=8)

# energy conservation
ax        = axes[1]
cons_true = (E_lep_true + E_roe_true) / E_nu_true.clip(min=1e-6)
cons_pred = (E_lep_pred + E_roe_pred) / E_nu_pred.clip(min=1e-6)
spread    = np.percentile(np.abs(np.concatenate([cons_true - 1, cons_pred - 1])), 99)
xlim_c    = min(spread * 1.5, 3.0)
bins_c    = np.linspace(max(0.0, 1 - xlim_c), 1 + xlim_c, 60)
ax.hist(cons_true, bins=bins_c, histtype="step", linewidth=1.5, label="True")
ax.hist(cons_pred, bins=bins_c, histtype="step", linewidth=1.5, linestyle="--",
        label="Predicted")
ax.axvline(1.0, color="k", linestyle=":", linewidth=1)
ax.set_xlabel("$(E_{lep} + E_{roe}) / E_\\nu$")
ax.set_ylabel("Events")
ax.set_title("Energy conservation")
ax.legend()
ax.text(0.05, 0.95,
        f"True  mean = {cons_true.mean():.3f}\nPred  mean = {cons_pred.mean():.3f}",
        transform=ax.transAxes, va="top", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))

plt.tight_layout()
plt.savefig(figures_path / "inelasticity_conservation.png", dpi=300)
plt.show()
print("Saved: inelasticity_conservation.png")

Poss future additions

- Best/worst event display: sort val_dataset by |pred − true| / true, visualise graph hits of best and worst events.
- Containment study: filter by vertex vz within detector acceptance. Check data.keys() and available parquet columns first.